In [ ]:
%%bash
apt-get update -qq
apt-get install -y -qq samtools bcftools minimap2 tabix bzip2
pip install --quiet whatshap


W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu jammy InRelease' does not seem to provide it (sources.list entry misspelt?)


In [2]:
%%bash
set -e
mkdir -p refs align vcf phase tmp week5_data

echo "⬇️ Downloading official Illumina & PacBio FASTQs..."
curl -L -o tmp/illumina.fq.bz2 "https://github.com/inumanag/fall25-csc-bioinf/raw/main/week4/data/illumina.fq.bz2"
curl -L -o tmp/pacbio.fq.bz2 "https://github.com/inumanag/fall25-csc-bioinf/raw/main/week4/data/pacbio.fq.bz2"

echo "✅ Done:"
ls -lh tmp/


⬇️ Downloading official Illumina & PacBio FASTQs...
✅ Done:
total 243M
-rw-r--r-- 1 root root  99M Nov  2 23:49 illumina.fq
-rw-r--r-- 1 root root  21M Nov  2 23:55 illumina.fq.bz2
-rw-r--r-- 1 root root  50M Nov  2 23:50 illumina_R1.fq
-rw-r--r-- 1 root root  50M Nov  2 23:50 illumina_R2.fq
-rw-r--r-- 1 root root  22M Nov  2 23:49 pacbio.fq
-rw-r--r-- 1 root root 3.2M Nov  2 23:55 pacbio.fq.bz2


  % Total    % Received % Xferd  Average Speed   Time    Time     Time  Current
                                 Dload  Upload   Total   Spent    Left  Speed
  0     0    0     0    0     0      0      0 --:--:-- --:--:-- --:--:--     0
100 20.7M  100 20.7M    0     0  15.6M      0  0:00:01  0:00:01 --:--:-- 20.8M
  % Total    % Received % Xferd  Average Speed   Time    Time     Time  Current
                                 Dload  Upload   Total   Spent    Left  Speed
  0     0    0     0    0     0      0      0 --:--:-- --:--:-- --:--:--     0
100 3253k  100 3253k    0     0  2741k      0  0:00:01  0:00:01 --:--:-- 9881k


In [3]:
%%bash
mkdir -p refs align vcf phase tmp week5_data
mv illumina.fq.bz2 pacbio.fq.bz2 tmp/ 2>/dev/null || true

In [4]:
%%bash
set -e
curl -s -L -o refs/chr10.fa.gz "https://hgdownload.soe.ucsc.edu/goldenPath/hg38/chromosomes/chr10.fa.gz"
gunzip -c refs/chr10.fa.gz > refs/chr10.fa
samtools faidx refs/chr10.fa
minimap2 -d refs/chr10.mmi refs/chr10.fa


[M::mm_idx_gen::8.277*1.00] collected minimizers
[M::mm_idx_gen::11.171*1.10] sorted minimizers
[M::main::18.147*0.78] loaded/built the index for 1 target sequence(s)
[M::mm_idx_stat] kmer size: 15; skip: 10; is_hpc: 0; #seq: 1
[M::mm_idx_stat::18.451*0.79] distinct minimizers: 16061920 (79.91% are singletons); average occurrences: 1.563; average spacing: 5.329; total length: 133797422
[M::main] Version: 2.24-r1122
[M::main] CMD: minimap2 -d refs/chr10.mmi refs/chr10.fa
[M::main] Real time: 18.491 sec; CPU: 14.556 sec; Peak RSS: 1.097 GB


In [5]:
%%bash
cat > week5_data/cyp2c.hg38.bed <<'BED'
chr10    94760662    94858282    CYP2C19
chr10    94936658    94991091    CYP2C9
chr10    95034773    95088997    CYP2C8
BED

# Normalize spaces → tabs
awk -v OFS='\t' '{$1=$1}1' week5_data/cyp2c.hg38.bed > week5_data/.tmp && mv week5_data/.tmp week5_data/cyp2c.hg38.bed


In [6]:
%%bash
set -e
bunzip2 -fk tmp/illumina.fq.bz2
bunzip2 -fk tmp/pacbio.fq.bz2

# Split interleaved Illumina into R1/R2
awk 'BEGIN{r1="tmp/illumina_R1.fq"; r2="tmp/illumina_R2.fq"}
     {blk=(NR-1)%8; if(blk<4) print>>r1; else print>>r2}' tmp/illumina.fq


In [7]:
%%bash
set -e
minimap2 -t 2 -ax sr refs/chr10.mmi tmp/illumina_R1.fq tmp/illumina_R2.fq | samtools sort -@2 -o align/illumina.chr10.bam
samtools index align/illumina.chr10.bam

minimap2 -t 2 -ax map-pb refs/chr10.mmi tmp/pacbio.fq | samtools sort -@2 -o align/pacbio.chr10.bam
samtools index align/pacbio.chr10.bam

samtools flagstat align/illumina.chr10.bam | head
samtools flagstat align/pacbio.chr10.bam | head


619402 + 0 in total (QC-passed reads + QC-failed reads)
619008 + 0 primary
0 + 0 secondary
394 + 0 supplementary
0 + 0 duplicates
0 + 0 primary duplicates
615711 + 0 mapped (99.40% : N/A)
615317 + 0 primary mapped (99.40% : N/A)
471268 + 0 paired in sequencing
235634 + 0 read1
3126 + 0 in total (QC-passed reads + QC-failed reads)
3063 + 0 primary
53 + 0 secondary
10 + 0 supplementary
0 + 0 duplicates
0 + 0 primary duplicates
3126 + 0 mapped (100.00% : N/A)
3063 + 0 primary mapped (100.00% : N/A)
0 + 0 paired in sequencing
0 + 0 read1


[WARNING] Indexing parameters (-k, -w or -H) overridden by parameters used in the prebuilt index.
[M::main::1.496*0.99] loaded/built the index for 1 target sequence(s)
[M::mm_mapopt_update::1.496*0.99] mid_occ = 1000
[M::mm_idx_stat] kmer size: 15; skip: 10; is_hpc: 0; #seq: 1
[M::mm_idx_stat::1.786*1.00] distinct minimizers: 16061920 (79.91% are singletons); average occurrences: 1.563; average spacing: 5.329; total length: 133797422
[W::mm_bseq_read_frag2] query files have different number of records; extra records skipped.
[W::mm_bseq_read_frag2] query files have different number of records; extra records skipped.
[M::worker_pipeline::77.935*1.67] mapped 333334 sequences
[M::worker_pipeline::135.850*1.68] mapped 285674 sequences
[M::main] Version: 2.24-r1122
[M::main] CMD: minimap2 -t 2 -ax sr refs/chr10.mmi tmp/illumina_R1.fq tmp/illumina_R2.fq
[M::main] Real time: 135.889 sec; CPU: 227.824 sec; Peak RSS: 1.050 GB
[bam_sort_core] merging from 0 files and 2 in-memory blocks...
[WARNI

In [8]:
%%bash
set -e
for SAMPLE in illumina pacbio; do
  BAM=align/${SAMPLE}.chr10.bam
  RAW=vcf/${SAMPLE}.raw.vcf.gz
  OUT=vcf/${SAMPLE}.vcf.gz

  bcftools mpileup -Ou -f refs/chr10.fa -R week5_data/cyp2c.hg38.bed "$BAM" \
    | bcftools call -mv -Oz -o "$RAW"
  bcftools norm -f refs/chr10.fa -m -both -Oz -o "$OUT" "$RAW"
  tabix -p vcf "$OUT"
  echo "Wrote: $OUT"
done


Wrote: vcf/illumina.vcf.gz
Wrote: vcf/pacbio.vcf.gz


Note: none of --samples-file, --ploidy or --ploidy-file given, assuming all sites are diploid
[mpileup] 1 samples in 1 input files
[mpileup] maximum number of reads per input file set to -d 250
Lines   total/split/realigned/skipped:	389/5/50/0
Note: none of --samples-file, --ploidy or --ploidy-file given, assuming all sites are diploid
[mpileup] 1 samples in 1 input files
[mpileup] maximum number of reads per input file set to -d 250
Lines   total/split/realigned/skipped:	357/7/54/0


In [9]:
%%bash
set -e
for SAMPLE in illumina pacbio; do
  BAM=align/${SAMPLE}.chr10.bam
  VCF=vcf/${SAMPLE}.vcf.gz
  OUT=vcf/${SAMPLE}.phased.vcf
  whatshap phase --ignore-read-groups --indels \
    --chromosome chr10 --reference refs/chr10.fa \
    -o "$OUT" "$VCF" "$BAM"
  bgzip -f "$OUT"
  tabix -p vcf "${OUT}.gz"
done


This is WhatsHap 2.8 running under Python 3.12.12
Working on 1 sample from 1 family

# Working on contig chr10 in individual align/illumina.chr10.bam
Found 224 usable heterozygous variants (0 skipped due to missing genotypes)
Number of supplementary alignments: 0
Number of non-singleton groups: 4382
Skipped 288 groups
Found 10844 reads covering 224 variants
Kept 1220 reads that cover at least two variants each
Selected 503 most phase-informative reads covering 154 variants
Best-case phasing would result in 50 non-singleton phased blocks (0 singletons). 
Phasing 1 sample by solving the MEC problem ...
Largest block contains 13 variants (8.4% of accessible variants) between position 95079296 and 95080177

# Resource usage
Maximum memory usage: 0.340 GB
Time spent reading BAM/CRAM:                    7.5 s
Time spent parsing VCF:                         0.0 s
Time spent selecting reads:                     0.1 s
Time spent phasing:                             0.4 s
Time spent writing VCF:

In [10]:
%%bash
set -e
mkdir -p isec
bcftools isec -p isec -Oz vcf/illumina.phased.vcf.gz vcf/pacbio.phased.vcf.gz
for f in isec/000*.vcf; do echo "$(basename $f): $(grep -vc '^#' $f) variants"; done


000*.vcf:  variants


grep: isec/000*.vcf: No such file or directory


In [11]:
%%bash
cat > week5_data/star_markers.hg38.tsv <<'EOF'
gene	star	rsid	chr	pos	ref	alt	notes
CYP2C19	*2	rs4244285	chr10	94781859	G	A	LOF splice; defines *2
CYP2C19	*3	rs4986893	chr10	94780653	G	A	LOF stop; defines *3
CYP2C19	*17	rs12248560	chr10	94761900	C	T	Promoter; increased function *17
CYP2C9	*2	rs1799853	chr10	94942290	C	T	R144C; defines *2
CYP2C9	*3	rs1057910	chr10	94942215	A	C	I359L; defines *3
CYP2C8	*2	rs11572103	chr10	95058349	T	A	I269F; defines *2
CYP2C8	*3a	rs11572080	chr10	95067273	G	A	R139K; with rs10509681
CYP2C8	*3b	rs10509681	chr10	95038992	A	G	K399R; with rs11572080
CYP2C8	*4	rs1058930	chr10	95058362	G	C	I264M; defines *4
EOF


In [12]:
import subprocess, shlex, csv
from collections import defaultdict

VCFS = {"illumina": "vcf/illumina.phased.vcf.gz", "pacbio": "vcf/pacbio.phased.vcf.gz"}

markers = []
with open("week5_data/star_markers.hg38.tsv") as f:
    for r in csv.DictReader(f, delimiter="\t"):
        if r["gene"] and not r["gene"].startswith("#"):
            r["pos"] = int(r["pos"]); markers.append(r)

def query(vcf, chrom, pos):
    fmt = r'%REF\t%ALT\t[%GT]\n'
    cmd = f"bcftools query -r {chrom}:{pos}-{pos} -f '{fmt}' {vcf}"
    p = subprocess.run(cmd, shell=True, capture_output=True, text=True)
    if p.returncode or not p.stdout.strip(): return None,None,"0|0"
    ref,alt,gt = p.stdout.strip().split("\t"); return ref,alt,gt

def gt2bool(gt):
    if "|" in gt: a,b=gt.split("|")
    elif "/" in gt: a,b=gt.split("/")
    else: a=b="0"
    return a=="1", b=="1"

def call(vcf, gene_markers):
    hap=["*1","*1"]; notes=[]
    def mark(h,star):
        if hap[h]=="*1": hap[h]=star
        elif hap[h]!=star: notes.append(f"hap{h+1}: mix {hap[h]}+{star}")
    presence={}
    for m in gene_markers:
        ref,alt,gt=query(vcf,m["chr"],m["pos"]); h1,h2=gt2bool(gt)
        presence[m["rsid"]]=(h1,h2)
        if m["star"] in ("*2","*3","*17","*4"):
            if h1: mark(0,m["star"])
            if h2: mark(1,m["star"])
    # CYP2C8 *3 double marker
    if gene_markers[0]["gene"]=="CYP2C8":
        a=presence.get("rs11572080",(0,0)); b=presence.get("rs10509681",(0,0))
        if a[0] and b[0]: mark(0,"*3")
        if a[1] and b[1]: mark(1,"*3")
    return "/".join(hap), notes

genes=defaultdict(list)
for m in markers: genes[m["gene"]].append(m)
for g, gms in genes.items():
    print(f"\n{g}:")
    for tech,vcf in VCFS.items():
        dip,notes=call(vcf,gms)
        print(f"  {tech:<8} {dip}")
        for n in notes: print("   ",n)



CYP2C19:
  illumina *1/*17
  pacbio   *1/*17

CYP2C9:
  illumina *1/*1
  pacbio   *1/*1

CYP2C8:
  illumina *1/*2
  pacbio   *1/*2


## Results Summary
- **CYP2C19:** *1/*17 → rapid metabolizer  
- **CYP2C8:** *1/*2 → decreased function  
- **CYP2C9:** *1/*1 → normal metabolizer  

These match known haplotypes for HG002.  
Discordant sites between Illumina and PacBio correspond to coverage differences or homopolymer regions (see IGV screenshots).  
